# Supplementary Material — 0. Data Acquisition

Fetches shot-level event data for the five 2015/16 European top leagues from
the [StatsBomb open data API](https://github.com/statsbomb/open-data) and
merges in pre-match Bet365 odds from [football-data.co.uk](https://www.football-data.co.uk/).
Each league is processed into a match-summary parquet file under
`../../data/input_data/` and consumed by `01_generate_probabilities.ipynb`.

**Required inputs:** internet access (StatsBomb) and the five league CSVs
from football-data.co.uk for season 2015/16, placed under `../../data/odds/`:

| League         | Football-data.co.uk file |
|----------------|--------------------------|
| Premier League | `E0.csv`                 |
| Bundesliga     | `D1.csv`                 |
| La Liga        | `SP1.csv`                |
| Serie A        | `I1.csv`                 |
| Ligue 1        | `F1.csv`                 |

Only matches for which both StatsBomb event data and Bet365 odds are available
are retained. For Ligue 1, StatsBomb provides 377 of the 380 regular-season matches;
one of those 377 cannot be matched to the football-data.co.uk odds file, leaving
376 complete Ligue 1 records. The other four leagues each have complete coverage,
yielding 1822 matches in total across the five leagues.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve() / 'modules'))

import pandas as pd

from data_loading import (
    get_available_competitions,
    load_statsbomb_competition,
    process_shots_to_match_summary,
    load_football_data_odds,
    merge_odds_with_matches,
    save_competition_data,
    TEAM_NAME_MAP_DEFAULT,
)

DATA_DIR  = Path('..') / '..' / 'data'
INPUT_DIR = DATA_DIR / 'input_data'
ODDS_DIR  = DATA_DIR / 'odds'
INPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SEASON = '2015/2016'

## 1. League configuration

Mapping from StatsBomb competition name → output slug + football-data.co.uk file.

In [2]:
LEAGUES = [
    {'comp': 'Premier League', 'slug': 'epl_1516',        'odds': 'E0.csv'},
    {'comp': '1. Bundesliga',  'slug': 'bundesliga_1516', 'odds': 'D1.csv'},
    {'comp': 'La Liga',        'slug': 'laliga_1516',     'odds': 'SP1.csv'},
    {'comp': 'Serie A',        'slug': 'seriea_1516',     'odds': 'I1.csv'},
    {'comp': 'Ligue 1',        'slug': 'ligue1_1516',     'odds': 'F1.csv'},
]
pd.DataFrame(LEAGUES)

,comp,slug,odds
0,Premier League,epl_1516,E0.csv
1,1. Bundesliga,bundesliga_1516,D1.csv
2,La Liga,laliga_1516,SP1.csv
3,Serie A,seriea_1516,I1.csv
4,Ligue 1,ligue1_1516,F1.csv


## 2. Resolve StatsBomb competition / season IDs

In [3]:
competitions = get_available_competitions()
league_ids = (
    competitions[
        competitions['competition_name'].isin([c['comp'] for c in LEAGUES]) &
        (competitions['season_name'] == TARGET_SEASON)
    ][['competition_id', 'season_id', 'competition_name', 'season_name']]
    .reset_index(drop=True)
)
league_ids

/home/max/drive/projects/4_forecasting-polynomial-multiplication/.venv/lib/python3.12/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


,competition_id,season_id,competition_name,season_name
0,9,27,1. Bundesliga,2015/2016
1,11,27,La Liga,2015/2016
2,7,27,Ligue 1,2015/2016
3,2,27,Premier League,2015/2016
4,12,27,Serie A,2015/2016


## 3. Load each league

For each league: download the season's matches and shot events from StatsBomb,
aggregate to one row per match, merge betting odds (when the CSV exists),
and write a parquet file to `../../data/input_data/`.

In [4]:
for cfg in LEAGUES:
    cname = cfg['comp']
    slug  = cfg['slug']
    print(f'\n=== {cname} ===')

    meta = league_ids[league_ids['competition_name'] == cname]
    if meta.empty:
        print(f'  Not found in StatsBomb open data for season {TARGET_SEASON}; skipping.')
        continue
    row = meta.iloc[0]

    shots, matches = load_statsbomb_competition(
        competition_id=int(row['competition_id']),
        season_id=int(row['season_id']),
        competition_name=cname,
    )
    summary = process_shots_to_match_summary(shots, matches)

    odds_path = ODDS_DIR / cfg['odds']
    if odds_path.exists():
        odds_df = load_football_data_odds(
            csv_path=odds_path,
            competition_name=cname,
            team_name_map=TEAM_NAME_MAP_DEFAULT,
            date_format='%Y-%m-%d',          # football-data.co.uk uses ISO dates here
        )
        summary = merge_odds_with_matches(summary, odds_df)
        # Retain only matches with complete records (both StatsBomb xG and Bet365 odds)
        n_before = len(summary)
        summary = summary[summary['betting_p_home'].notna()].reset_index(drop=True)
        if len(summary) < n_before:
            print(f'  Retained {len(summary)} complete records (dropped {n_before - len(summary)} without odds).')
    else:
        print(f'  No odds CSV at {odds_path}; saving without odds.')

    save_competition_data(summary, INPUT_DIR, competition_slug=slug)


=== Premier League ===


Premier League:   0%|          | 0/380 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 4. Saved files

In [ ]:
rows = []
for cfg in LEAGUES:
    f = INPUT_DIR / f"{cfg['slug']}.parquet"
    if not f.exists():
        rows.append({'file': f.name, 'matches': 0, 'has_odds': 0, 'season': '—'})
        continue
    df = pd.read_parquet(f)
    rows.append({
        'file':     f.name,
        'matches':  len(df),
        'has_odds': int(df['betting_p_home'].notna().sum()) if 'betting_p_home' in df.columns else 0,
        'season':   df['season_name'].iloc[0] if 'season_name' in df.columns else '?',
    })
pd.DataFrame(rows)